In [1]:
import sys

sys.path.append("/mnt/d/work/smolagents/src")

In [3]:
from google import genai
from dotenv import load_dotenv
from typing import Optional, Dict, List, Callable
import requests
import os
from smolagents import (
    ApiModel,
    Tool,
    ChatMessage,
    ToolCallingAgent,
    DuckDuckGoSearchTool,
    OpenAIServerModel,
    tool,
    
)
from google.genai import types
import json
import uuid
from smolagents.models import ChatMessageToolCall,ChatMessageToolCallDefinition
from PIL import Image

In [4]:
load_dotenv()

True

In [4]:
@tool
def get_weather(city: str) -> str:
    """
    Get the weather for a given city.
    Args:
        city (str): The name of the city.
    """
    # This is a placeholder function. You would replace this with actual weather data retrieval logic.
    return f"The weather in {city} is sunny."

In [5]:
def get_gemini_tool(tool:Tool):
    name=tool.name
    description=tool.description
    inputs=tool.inputs
    required=[]
    for arg,desc in inputs.items():
        if 'nullable' in desc:
            desc.pop('nullable')
        else:
            required.append(arg)
        if 'type' in desc and desc['type'] == 'any':
           desc['type'] = 'object'
    output={"name":name,"description":description,"parameters":{"type":"object","properties":inputs,"required":required}}
    return output



In [6]:
class GeminiModel(ApiModel):
    def __init__(
        self,
        model_id: str = "gemini-2.0-flash",
        timeout: Optional[int] = 120,
        custom_role_conversions: Optional[Dict[str, str]] = None,
        **kwargs,
    ):
        from google import genai

        super().__init__(**kwargs)
        self.client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
        self.model_id = model_id
        self.timeout = timeout
        self.custom_role_conversions = custom_role_conversions

    def __call__(
        self,
        messages: List[Dict[str, str]],
        stop_sequences: Optional[List[str]] = None,
        grammar: Optional[str] = None,
        tools_to_call_from: Optional[List[Tool]] = None,
        **kwargs,
    ) -> ChatMessage:
        # print("messages", messages)
        messages_part = []
        system_instruction = None
        last_function_call = None
        for message in messages:
            if message["role"] == "system":
                system_instruction = message["content"][0]["text"]
            if message["role"] == "user":
                parts= []
                for part in message["content"]:
                    if part["type"] == "text":
                        parts.append(types.Part.from_text(text=part["text"]))
                        messages_part.append(
                    types.Content(
                        parts=parts,
                        role="user"
                    )
                )
                    elif part["type"] == "image":
                        messages_part.append(part["image"])
                
            if message["role"] == "tool-call":
                text = message["content"][0]["text"].strip()
                text = text.replace("Calling tools:\n", "").replace("'", '"')
                text = json.loads(text)[0]
                last_function_call = text["function"]["name"]
                messages_part.append(
                    types.Content(
                        parts=[
                            types.Part.from_function_call(
                                name=text["function"]["name"],
                                args=text["function"]["arguments"],
                            )
                        ]
                    )
                )
            if message["role"] == "tool-response":
                messages_part.append(
                    types.Content(
                        parts=[
                            types.Part.from_function_response(
                                name=last_function_call,
                                response={"result": message["content"][0]["text"]},
                            )
                        ]
                    )
                )
        tools_list = None
        if tools_to_call_from:
            tools_list = []
            for tool_ in tools_to_call_from:
                tools_list.append(get_gemini_tool(tool_))
        tools_list = types.Tool(function_declarations=tools_list)
        generation_config = types.GenerateContentConfig(
            tools=[tools_list],
            stop_sequences=stop_sequences,
            system_instruction="you are a helpful tool calling agent. You can solve the user query by calling the tools. You are given access to some tools. when you feel you have the answer of the users query you have to call final_answer tool",
        )
        # print(messages_part)
        response = self.client.models.generate_content(
            model=self.model_id, contents=messages_part, config=generation_config
        )
        print(response)
        output = ChatMessage(role="assistant", raw=response)
        # print(response)
        for part in response.candidates[0].content.parts:
            if part.function_call:
                function_call = part.function_call
                function_name = function_call.name
                arguments = function_call.args
                output.tool_calls = [
                    ChatMessageToolCall(
                        function=ChatMessageToolCallDefinition(
                            name=function_name, arguments=arguments
                        ),
                        id=str(uuid.uuid4()),
                        type="function"
                    )
                ]
            if part.text:
                output.text = part.text

        return output

In [5]:
@tool
def get_weather(location: str) -> dict:
    """this function takes the location and returns the weather statistics:
    Args:
        location: place for which we want the weather statistics like temperature, weather description,wind speed, wind degree, wind direction,pressure
                  precipitation,humidity,cloud cover ,feels like ,uv index and visibility.
    """
    access_key = os.getenv("WEATHER_API_KEY")
    endpoint = f"http://api.weatherstack.com/current?access_key={access_key}&query={location}"
    resp = requests.get(endpoint)
    return resp.json()


In [8]:
sentiment_tool = Tool.from_space(space_id="AventIQ-AI/Sentiment-Analysis",token=os.getenv("HF_TOKEN"),
                                 description="This tool takes the text and returns the sentiment of the text",name="sentiment_tool")

Loaded as API: https://aventiq-ai-sentiment-analysis.hf.space ✔


Since `api_name` was not defined, it was automatically set to the first available API: `/predict`.


In [9]:
image_caption_tool=Tool.from_space(space_id="fancyfeast/joy-caption-alpha-two",
                                   token=os.getenv("HF_TOKEN"),
                                 description="This tool takes the image and returns the caption of the image",name="image_caption_tool")

Loaded as API: https://fancyfeast-joy-caption-alpha-two.hf.space ✔


Since `api_name` was not defined, it was automatically set to the first available API: `/stream_chat`.


In [10]:
get_gemini_tool(image_caption_tool)

{'name': 'image_caption_tool',
 'description': 'This tool takes the image and returns the caption of the image',
 'parameters': {'type': 'object',
  'properties': {'input_image': {'type': 'object', 'description': ''},
   'name_input': {'type': 'string', 'description': ''},
   'custom_prompt': {'type': 'string', 'description': ''}},
  'required': ['input_image', 'name_input', 'custom_prompt']}}

In [11]:
model1=GeminiModel(model_id="gemini-2.0-flash", timeout=120)
# model2=OpenAIServerModel(model_id="gpt-4", timeout=120,api_key=os.getenv("OPENAI_API_KEY"))

In [7]:
import importlib
from typing import Any

In [11]:
class GeminiModel(OpenAIServerModel):
    def __init__(
        self,
        model_id: str,
        api_key: Optional[str] = None,
        client_kwargs: Optional[Dict[str, Any]] = None,
        custom_role_conversions: Optional[Dict[str, str]] = None,
        **kwargs,
    ):
        if importlib.util.find_spec("openai") is None:
            raise ModuleNotFoundError(
                "Please install 'openai' extra to use AzureOpenAIServerModel: `pip install 'smolagents[openai]'`"
            )
        client_kwargs = client_kwargs or {}
        super().__init__(
            model_id=model_id,
            api_key=api_key,
            api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
            custom_role_conversions=custom_role_conversions,
            **kwargs,
        )


In [12]:
model=GeminiModel(model_id="gemini-2.0-flash",api_key=os.getenv("GOOGLE_API_KEY"))

In [14]:
agent=ToolCallingAgent(model=model,tools=[get_weather])

In [15]:
agent.run("what is the temperature in hyderabad india")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ what is the temperature in hyderabad india                                                                      │
│                                                                                                                 │
╰─ GeminiModel - gemini-2.0-flash ────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'get_weather' with arguments: {'location': 'hyderabad india'}                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {'request': {'type': 'City', 'query': 'Hyderabad, India', 'language': 'en', 'unit': 'm'}, 'location':
{'name': 'Hyderabad', 'country': 'India', 'region': 'Telangana', 'lat': '17.375', 'lon': '78.474', 'timezone_id': 
'Asia/Kolkata', 'localtime': '2025-05-02 11:12', 'localtime_epoch': 1746184320, 'utc_offset': '5.50'}, 'current': 
{'observation_time': '05:42 AM', 'temperature': 32, 'weather_code': 116, 'weather_icons': 
|'https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0002_sunny_intervals.png'], 
'weather_descriptions': |'Partly cloudy'], 'astro': {'sunrise': '05:50 AM', 'sunset': '06:36 PM', 'moonrise': 
'09:53 AM', 'moonset': '11:39 PM', 'moon_phase': 'Waxing Crescent', 'moon_illumination': 24}, 'air_quality': {'co':
'569.8', 'no2': '14.615', 'o3': '75', 'so2': '11.285', 'pm2_5': '35.52', 'pm10': '45.14', 'us-epa-index': '2', 
'gb-defra-index': '2'}, 'wind_speed': 5, 'wind_degree': 203, 'wind_dir': 'SSW', 'pressure': 1013, 'precip': 0, 
'humidity': 56, 'cloudcover': 50, 'feelslike': 31, 'uv_index': 10, 'visibility': 6, 'is_day': 'yes'}}

[Step 1: Duration 3.60 seconds| Input tokens: 1,095 | Output tokens: 7]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The temperature in Hyderabad, India is 32 degrees      │
│ Celsius'}                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Final answer: The temperature in Hyderabad, India is 32 degrees Celsius

[Step 2: Duration 0.94 seconds| Input tokens: 2,702 | Output tokens: 23]

'The temperature in Hyderabad, India is 32 degrees Celsius'